# Phase 3 — first end-to-end training of the stitched bridge

Plan v2 (`docs/encoder_decoder_bridge_e2e_plan_v2.md`), Phase 3; implementation notes in `docs/stitched-e2e-plan.md`.

```
English source ──► ModernBERT-base + LoRA ──► bridge: Linear 768→2560 + RMSNorm (no bias)          [GPU 0]
                                                     │ memory + padding mask
                                                     ▼
persona instruction ──► Gemma 3 4B IT (4-bit) + LoRA                                               [GPU 1]
  (no source text)        after every 4th layer: h ← h + tanh(gate) · CrossAttention(norm(h), memory)   (no bias)
```

**What Phase 2 established** (`5-bridge-skeleton/`, three runs): the plumbing is correct (gate-0 equivalence, padding mask, generation with the cache, gradients in all five groups); with the shortcuts closed (no bias terms, nothing to memorise) the bridge **does** carry content — on random-word copying the gap between the right and another sentence's memory rose steadily (0.003 → 0.15 in 300 steps), and persona outputs then changed exactly with the memory — but with 120 persona pairs and frozen LoRAs the decoder did not learn to *use* the content for meaning.

**This notebook:**

- **Data:** v4 merged (`data/persona_translation_v4/merged/`): **768 training pairs**; checkpoint chosen on **validation** (75 pairs); **test** (75), **out-of-domain** (114) and **minimal pairs** (72) generated once at the end.
- **Stage A — copy warm-up** (bridge, cross-attention, gates only): 400 steps of copying random word sequences, which starts the content flow (Phase 2, run 3).
- **Stage B — full training:** encoder LoRA, bridge, cross-attention, gates **and** decoder LoRA, base weights frozen (plan v2 Phase 3). Learning rates: decoder LoRA 5e-5 (Phase 1 over-fitted at 1e-4), encoder LoRA 1e-4, bridge and cross-attention 3e-4, gates 1e-2. Up to 5 epochs, early stopping after 2 epochs without a lower validation loss, best epoch kept.
- **Each epoch:** training loss; validation loss with the right memory, another sentence's memory, and none; the **gap** (other − right); gate values; gradient norms per group; probe outputs.

**Phase 3 gate** (plan v2, made measurable):

| # | Check | Pass |
|---|---|---|
| 1 | All five groups train | non-zero gradient norm in every group, every epoch |
| 2 | Training works without degenerate output | training loss falls; on validation: stops ≥ 95 %, repetition ≤ 5 % |
| 3 | **The decoder uses the source through the bridge** | validation gap (other − right) ≥ 0.1, and chrF with the right memory at least 2 points above chrF with another sentence's memory |
| 4 | Minimal pairs | the two sentences of a pair give different outputs in ≥ 80 % of cases; numbers carried in ≥ 80 % of number pairs |

Meaning preservation and persona recognition are judged in Phase 4/6, against the direct-source baseline; this notebook reports chrF and samples so the first reading is possible.

In [1]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup

Two GPUs (Kaggle: *GPU T4 × 2*): encoder and bridge on `cuda:0`, decoder and cross-attention on `cuda:1`; fp32 compute, 4-bit decoder weights.

In [2]:
# Cell 1 — Packages, settings, versions, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, math, random, re, statistics, time, urllib.request
from collections import Counter
from itertools import combinations

import bitsandbytes
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v4/merged/")
XATTN_EVERY, XATTN_HEADS = 4, 8
MAX_NEW_TOKENS = 80
SEED = 42
DTYPE = torch.float32

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)

api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "revisions": REV}
print(json.dumps(VERSIONS, indent=1))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 84.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.8 MB/s eta 0:00:00:00:0100:01
{
 "torch": "2.11.0+cu128",
 "transformers": "5.17.0",
 "peft": "0.21.2",
 "bitsandbytes": "0.50.2",
 "gpus": [
  "Tesla T4",
  "Tesla T4"
 ],
 "revisions": {
  "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
  "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767"
 }
}


## Cell 2 — Data v4 (merged) and the records

Every example becomes a record: **prompt** (the persona instruction, never the sentence), **target**, **source** (what the encoder reads) and **references** (all acceptable targets, for scoring). The cell checks that no training sentence appears in an evaluation split.

In [3]:
# Cell 2 — Load the merged v4 splits; persona instructions; records

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
PERSONAS = list(MANIFEST["personas"])
INSTRUCTION = ("Translate the English sentence you are given into natural Roman-script Hinglish in the voice of "
               "{name}. Persona: {description} Preserve the original meaning; do not add new events or facts.")


def record(row):
    persona = MANIFEST["personas"][row["persona_id"]]
    return {**row, "persona": row["persona_id"], "prompt": INSTRUCTION.format(**persona)}


SPLITS = {s: [record(r) for r in fetch_jsonl(f"{s}.jsonl")]
          for s in ("train", "validation", "test", "ood", "minimal_pairs")}
TRAIN, VAL = SPLITS["train"], SPLITS["validation"]
train_text = {r["source"].lower() for r in TRAIN}
assert not any(r["source"].lower() in train_text for s in ("validation", "test", "ood", "minimal_pairs")
               for r in SPLITS[s]), "an evaluation sentence appears in training"
assert not any(r["source"] in r["prompt"] for rs in SPLITS.values() for r in rs)

print({s: len(rs) for s, rs in SPLITS.items()}, "| personas", PERSONAS)
print("encoder input:", TRAIN[0]["source"])
print("decoder prompt:", TRAIN[0]["prompt"])
print("target:", TRAIN[0]["target"])

{'train': 768, 'validation': 75, 'test': 75, 'ood': 114, 'minimal_pairs': 72} | personas ['don', 'anthony_gonsalves', 'jai_sholay']
encoder input: I don't trust someone who changes their story every time.
decoder prompt: Translate the English sentence you are given into natural Roman-script Hinglish in the voice of Don. Persona: Cool, calculating and faintly amused; short, controlled sentences; treats every situation as a move in a game; a quiet hint of control, never loud. Preserve the original meaning; do not add new events or facts.
target: Jo har baar kahani badalta hai, uspe bharosa nahi. Simple.


## Cell 3 — Load the encoder (GPU 0) and the decoder (GPU 1)

Stops with a restart message if the GPUs already hold models from an earlier run in the same kernel.

In [4]:
# Cell 3 — ModernBERT-base on GPU 0; Gemma 3 4B IT (4-bit NF4, fp32 compute) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError(f"GPU memory already in use ({used[ENC_DEVICE]:.1f} GB / {used[DEC_DEVICE]:.1f} GB): restart "
                       "the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE).eval()

dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
TEXT = decoder.config.text_config
D_ENC, D_DEC, N_LAYERS = encoder.config.hidden_size, TEXT.hidden_size, TEXT.num_hidden_layers
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set(eos if isinstance(eos, list) else [eos]))
PAD = dec_tok.pad_token_id
print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | stop tokens {STOPS} | "
      f"GPU 0 {torch.cuda.memory_allocated(ENC_DEVICE) / 2**30:.1f} GB, "
      f"GPU 1 {torch.cuda.memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

encoder width 768 | decoder width 2560, 34 layers | stop tokens [1, 106] | GPU 0 0.6 GB, GPU 1 4.5 GB


## Cell 4 — Bridge, gated cross-attention (no biases), hooks, batch helpers

As in Phase 2, run 3: no bias terms anywhere in the bridge or the cross-attention, so everything they add comes from the memory; gates start at 0.

In [5]:
# Cell 4 — Bridge, gated cross-attention, hooks, encoder and decoder batch helpers

class Bridge(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=False)
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.proj(h))


class GatedCrossAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, h, memory, pad):
        out, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=pad, need_weights=False)
        return h + torch.tanh(self.gate) * out


class Context:
    enabled, memory, pad, calls = True, None, None, 0


CTX = Context()
XATTN_LAYERS = list(range(XATTN_EVERY - 1, N_LAYERS, XATTN_EVERY))
torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
xattn = nn.ModuleDict({str(i): GatedCrossAttention(D_DEC, XATTN_HEADS) for i in XATTN_LAYERS}).to(DEC_DEVICE)


def make_hook(i):
    block = xattn[str(i)]

    def hook(module, args, output):
        if not CTX.enabled:
            return output
        CTX.calls += 1
        h = output[0] if isinstance(output, tuple) else output
        h = block(h, CTX.memory, CTX.pad)
        return (h, *output[1:]) if isinstance(output, tuple) else h

    return hook


HOOKS = [LAYERS[i].register_forward_hook(make_hook(i)) for i in XATTN_LAYERS]


def encode(sources, train_encoder=False):
    t = enc_tok(sources, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    with torch.set_grad_enabled(train_encoder and torch.is_grad_enabled()):
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    return bridge(h).to(DEC_DEVICE), (t["attention_mask"] == 0).to(DEC_DEVICE)


def set_memory(memory, pad, enabled=True):
    CTX.memory, CTX.pad, CTX.enabled = memory, pad, enabled


def prompt_text(prompt):
    return dec_tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                       tokenize=False)


def encode_pair(r):
    msgs = [{"role": "user", "content": r["prompt"]}, {"role": "assistant", "content": r["target"]}]
    full, head = dec_tok.apply_chat_template(msgs, tokenize=False), prompt_text(r["prompt"])
    assert full.startswith(head)
    prompt = dec_tok(head, add_special_tokens=False)["input_ids"]
    reply = dec_tok(full[len(head):], add_special_tokens=False)["input_ids"]
    assert any(t in STOPS for t in reply)
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def decoder_batch(records):
    pairs = [encode_pair(r) for r in records]
    T = max(len(x) for x, _ in pairs)
    ids = torch.full((len(pairs), T), PAD)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    return {"input_ids": ids.to(DEC_DEVICE), "attention_mask": mask.to(DEC_DEVICE), "labels": labels.to(DEC_DEVICE),
            "token_type_ids": torch.zeros_like(ids).to(DEC_DEVICE)}


print(f"cross-attention after layers {XATTN_LAYERS}")

cross-attention after layers [3, 7, 11, 15, 19, 23, 27, 31]


## Cell 5 — LoRA at both ends, parameter groups, quick plumbing checks

Encoder LoRA (ModernBERT `Wqkv`, `Wo`, `Wi`) and decoder LoRA (Gemma attention and MLP projections, language model only), rank 16. Two quick checks from Phase 2 confirm the plumbing in this kernel: with gates at 0 the logits equal plain Gemma's, and every cross-attention block runs.

In [6]:
# Cell 5 — LoRA, parameter groups, gate-0 and hook checks

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["Wqkv", "Wo", "Wi"]))
assert all(LAYERS[i]._forward_hooks for i in XATTN_LAYERS)

GROUPS = {
    "encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
    "bridge": list(bridge.parameters()),
    "cross-attention": [p for n, p in xattn.named_parameters() if not n.endswith("gate")],
    "gates": [b.gate for b in xattn.values()],
    "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n],
}


def set_trainable(*names):
    for name, params in GROUPS.items():
        for p in params:
            p.requires_grad_(name in names)


for name, params in GROUPS.items():
    print(f"  {name:16s} {sum(p.numel() for p in params) / 1e6:8.2f} M parameters")

dec_model.eval()
with torch.no_grad():
    dec = decoder_batch(VAL[:2])
    dec.pop("labels")
    memory, pad = encode([r["source"] for r in VAL[:2]])
    set_memory(memory, pad, enabled=False)
    plain = dec_model(**dec).logits[:, -16:].clone()
    set_memory(memory, pad, enabled=True)
    CTX.calls = 0
    stitched = dec_model(**dec).logits[:, -16:].clone()
CHECKS = {"gate-0 equivalence": (plain - stitched).abs().max().item() <= 1e-5,
          "hooks fire": CTX.calls == len(XATTN_LAYERS)}
del plain, stitched, memory, pad
torch.cuda.empty_cache()
print(CHECKS)
assert all(CHECKS.values()), "plumbing check failed: see Phase 2"

  encoder LoRA         3.38 M parameters
  bridge               1.97 M parameters
  cross-attention    209.74 M parameters
  gates                0.00 M parameters
  decoder LoRA        29.80 M parameters
{'gate-0 equivalence': True, 'hooks fire': True}


## Cell 6 — Losses, generation and training helpers

- `losses(records)`: validation loss with the **right** memory, **another** sentence's memory (the next source id, cyclic — always a different sentence), and **none**; `gap` = other − right.
- `generate(record, memory_source)`: greedy, 80 new tokens at most, no repetition guard.
- `snapshot()` / `restore()`: copy of all trainable tensors (both LoRAs, bridge, cross-attention, gates) on the CPU, to keep the best epoch.

In [7]:
# Cell 6 — Losses (right / other / none), generation, snapshots, optimizer

def stitched_loss(records, memory_sources=None, enabled=True, train_encoder=False):
    dec = decoder_batch(records)
    memory, pad = encode(memory_sources or [r["source"] for r in records], train_encoder)
    set_memory(memory, pad, enabled)
    return dec_model(**dec).loss


def other_sources(records):
    text = {r["source_id"]: r["source"] for r in records}
    ids = sorted(text)
    nxt = {sid: ids[(k + 1) % len(ids)] for k, sid in enumerate(ids)}
    return [text[nxt[r["source_id"]]] for r in records]


@torch.no_grad()
def losses(records, batch=3):
    dec_model.eval()
    encoder.eval()
    others = other_sources(records)
    out = {"right": 0.0, "other": 0.0, "none": 0.0}
    for i in range(0, len(records), batch):
        chunk, oth = records[i:i + batch], others[i:i + batch]
        out["right"] += stitched_loss(chunk).item() * len(chunk)
        out["other"] += stitched_loss(chunk, memory_sources=oth).item() * len(chunk)
        out["none"] += stitched_loss(chunk, enabled=False).item() * len(chunk)
    out = {k: round(v / len(records), 4) for k, v in out.items()}
    out["gap"] = round(out["other"] - out["right"], 4)
    return out


@torch.no_grad()
def generate(r, memory_source=None, max_new_tokens=MAX_NEW_TOKENS):
    dec_model.eval()
    encoder.eval()
    ids = dec_tok(prompt_text(r["prompt"]), return_tensors="pt", add_special_tokens=False).to(DEC_DEVICE)
    memory, pad = encode([memory_source or r["source"]])
    set_memory(memory, pad, True)
    out = dec_model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=max_new_tokens,
                             eos_token_id=STOPS, pad_token_id=PAD, temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    return {"output": dec_tok.decode(new, skip_special_tokens=True).strip(), "new_tokens": len(new),
            "ended": "stop token" if new and new[-1] in STOPS else "token limit"}


def gate_values():
    return [round(math.tanh(b.gate.item()), 3) for b in xattn.values()]


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


def group_grad_norms():
    return {name: round(math.sqrt(sum(float(p.grad.norm()) ** 2 for p in params if p.grad is not None)), 4)
            for name, params in GROUPS.items()}


def show_probes(records, label):
    for r in records:
        print(f"  [{label}] {r['persona']:18s} {generate(r)['output']}")

## Cell 7 — Stage A: copy warm-up (bridge, cross-attention, gates)

400 steps of copying random word sequences (5–10 words from the training sources' vocabulary), fresh every batch — nothing to memorise, so the loss can only fall by reading the memory. Every 100 steps: the three losses and the gap on 30 held-out random sequences.

In [8]:
# Cell 7 — Stage A: random-word copy warm-up

COPY_INSTRUCTION = "Repeat the words you are given, in the same order."
WORDS = sorted({w for r in TRAIN for w in re.findall(r"[a-z]+", r["source"].lower()) if len(w) > 2})


def copy_record(rng, key):
    words = " ".join(rng.choice(WORDS) for _ in range(rng.randint(5, 10)))
    return {"source_id": f"rand{key}", "persona": None, "source": words, "target": words, "prompt": COPY_INSTRUCTION}


COPY_VAL = [copy_record(random.Random(SEED + 1000 + k), f"{k:02d}") for k in range(30)]
WARMUP_STEPS, WARMUP_EVAL, MICRO, ACCUM = 400, 100, 2, 2
set_trainable("bridge", "cross-attention", "gates")
opt = torch.optim.AdamW([{"params": GROUPS["bridge"] + GROUPS["cross-attention"], "lr": 3e-4},
                         {"params": GROUPS["gates"], "lr": 1e-2}], weight_decay=0.0)
WARMUP_LOG = [{"step": 0, **losses(COPY_VAL)}]
rng, t0, run = random.Random(SEED), time.perf_counter(), []
torch.cuda.reset_peak_memory_stats(ENC_DEVICE)
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)
for step in range(1, WARMUP_STEPS + 1):
    dec_model.train()
    for _ in range(ACCUM):
        loss = stitched_loss([copy_record(rng, f"t{step}") for _ in range(MICRO)])
        (loss / ACCUM).backward()
        run.append(loss.item())
    opt.step()
    opt.zero_grad()
    if step % WARMUP_EVAL == 0:
        WARMUP_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4), **losses(COPY_VAL),
                           "gates": gate_values()})
        run = []
        print(WARMUP_LOG[-1], f"| {time.perf_counter() - t0:.0f}s")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{'step': 100, 'train_loss': 7.3074, 'right': 6.3523, 'other': 6.363, 'none': 18.1847, 'gap': 0.0107, 'gates': [-0.049, -0.06, 0.118, 0.149, -0.141, -0.171, -0.237, 0.419]} | 220s
{'step': 200, 'train_loss': 6.0299, 'right': 6.0063, 'other': 6.1104, 'none': 18.1847, 'gap': 0.1041, 'gates': [-0.062, -0.065, 0.132, 0.128, -0.129, -0.179, -0.251, 0.495]} | 443s
{'step': 300, 'train_loss': 5.859, 'right': 5.8217, 'other': 6.0043, 'none': 18.1847, 'gap': 0.1826, 'gates': [-0.045, -0.068, 0.134, 0.131, -0.143, -0.167, -0.245, 0.547]} | 663s
{'step': 400, 'train_loss': 5.6897, 'right': 5.7235, 'other': 5.9819, 'none': 18.1847, 'gap': 0.2584, 'gates': [-0.059, -0.069, 0.127, 0.142, -0.142, -0.149, -0.252, 0.574]} | 883s


## Cell 8 — Stage B: full training (both LoRAs, bridge, cross-attention, gates)

768 training pairs, micro-batches of 2 with 2-step accumulation, up to 5 epochs; early stopping after 2 epochs without a lower validation loss (right memory); the best epoch's tensors are kept on the CPU. After each epoch: training loss, validation losses and gap, gate values, gradient norms per group (from the epoch's last step), and probe outputs (two validation sources × three personas).

In [9]:
# Cell 8 — Stage B: train all five groups; validation each epoch; keep the best epoch

MAX_EPOCHS, PATIENCE = 5, 2
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "cross-attention": 3e-4, "gates": 1e-2, "decoder LoRA": 5e-5}
set_trainable(*GROUPS)
opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": lr} for g, lr in LRS.items()], weight_decay=0.0)
steps_per_epoch = -(-len(TRAIN) // (MICRO * ACCUM))
total_steps = MAX_EPOCHS * steps_per_epoch
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / total_steps)
probe_ids = sorted({r["source_id"] for r in VAL})[:2]
PROBES = [r for r in VAL if r["source_id"] in probe_ids]

LOG = [{"epoch": 0, "validation": losses(VAL), "gates": gate_values()}]
print(LOG[-1])
best = (LOG[0]["validation"]["right"], 0, snapshot())
since_best, rng, t0 = 0, random.Random(SEED), time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    dec_model.train()
    encoder.train()
    order = rng.sample(TRAIN, len(TRAIN))
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    run = []
    for n, b in enumerate(batches, 1):
        loss = stitched_loss(b, train_encoder=True)
        assert torch.isfinite(loss), "non-finite loss"
        (loss / ACCUM).backward()
        run.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            norms = group_grad_norms()
            opt.step()
            sched.step()
            opt.zero_grad()
    LOG.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4), "validation": losses(VAL),
                "gates": gate_values(), "grad_norms": norms})
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")
    show_probes(PROBES, f"epoch {epoch}")
    if LOG[-1]["validation"]["right"] < best[0]:
        best, since_best = (LOG[-1]["validation"]["right"], epoch, snapshot()), 0
    else:
        since_best += 1
    if since_best >= PATIENCE:
        print(f"early stop: no lower validation loss for {PATIENCE} epochs")
        break

restore(best[2])
BEST_EPOCH = best[1]
TRAIN_SECONDS = round(time.perf_counter() - t0)
PEAK = {"GPU 0": round(torch.cuda.max_memory_allocated(ENC_DEVICE) / 2**30, 2),
        "GPU 1": round(torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30, 2)}
print(f"kept epoch {BEST_EPOCH} (validation loss, right memory {best[0]}) | peak memory {PEAK}")

{'epoch': 0, 'validation': {'right': 8.5627, 'other': 8.6136, 'none': 9.454, 'gap': 0.0509}, 'gates': [-0.059, -0.069, 0.127, 0.142, -0.142, -0.149, -0.252, 0.574]}
{'epoch': 1, 'train_loss': 3.4237, 'validation': {'right': 3.6891, 'other': 3.7388, 'none': 3.9692, 'gap': 0.0497}, 'gates': [-0.019, -0.03, 0.088, 0.116, -0.219, -0.07, -0.209, 0.501], 'grad_norms': {'encoder LoRA': 0.2855, 'bridge': 0.2368, 'cross-attention': 0.3207, 'gates': 0.3607, 'decoder LoRA': 7.6418}} | 878s
  [epoch 1] don                Blue folder mein original file hai.
  [epoch 1] anthony_gonsalves  Blue folder ko green folder ka replacement mat karo, warna file ka track lost ho jaayega!
  [epoch 1] jai_sholay         Farah ka laptop use karne se pehle permission chahiye.
  [epoch 1] don                Farah ka laptop use karne se pehle permission chahiye.
  [epoch 1] anthony_gonsalves  Blue folder ko Monday ko return kar do, warna green folder ko bhi koach kar dena!
  [epoch 1] jai_sholay         Farah ka lap

## Cell 9 — Generate with the kept checkpoint

Right memory on validation, test, out-of-domain and minimal pairs; **another sentence's memory** on validation as well, for the memory-dependence check. Test is generated here once and only reported.

In [10]:
# Cell 9 — Generation: every evaluation split with the right memory; validation also with another sentence's memory

RESULTS = []
t0 = time.perf_counter()
for split in ("validation", "test", "ood", "minimal_pairs"):
    for r in SPLITS[split]:
        RESULTS.append({"split": split, "memory": "right", "id": r["id"], "source_id": r["source_id"],
                        "persona": r["persona"], "source": r["source"], "references": r["references"],
                        "pair_id": r.get("pair_id"), **generate(r)})
for r, oth in zip(VAL, other_sources(VAL)):
    RESULTS.append({"split": "validation", "memory": "other", "id": r["id"], "source_id": r["source_id"],
                    "persona": r["persona"], "source": r["source"], "memory_of": oth, "references": r["references"],
                    "pair_id": None, **generate(r, memory_source=oth)})
print(f"{len(RESULTS)} outputs in {time.perf_counter() - t0:.0f}s")

411 outputs in 1065s


## Cell 10 — Measures and the Phase 3 gate

- **Stopping, repetition** (a word 4-gram twice), **Devanagari**, **chrF** against the best of the references (character n-grams 1–6, β = 2, computed here), **persona overlap** (mean word Jaccard between the three personas' outputs for a source; the references' own overlap is the yardstick).
- **Memory dependence** (validation): share of outputs that change when another sentence's memory is given, and chrF with the right vs the other memory.
- **Minimal pairs:** share of (pair, persona) whose two outputs differ; for pairs whose sentences differ in a number, share whose outputs carry the right number.

In [11]:
# Cell 10 — Measures per split; memory dependence; minimal pairs; the gate

def words(t):
    return re.findall(r"[a-z0-9']+", t.lower())


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, rr = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * rr / (beta ** 2 * p + rr) if p + rr else 0.0


def best_chrf(r):
    return max(chrf(r["output"], ref) for ref in r["references"])


def repetition(t):
    w = words(t)
    grams = [" ".join(w[i:i + 4]) for i in range(len(w) - 3)]
    return len(grams) != len(set(grams))


def overlap(rows):
    groups = {}
    for r in rows:
        groups.setdefault(r["source_id"], []).append(r["output"])
    vals = [statistics.mean(len(set(words(a)) & set(words(b))) / max(1, len(set(words(a)) | set(words(b))))
                            for a, b in combinations(v, 2)) for v in groups.values() if len(v) > 1]
    return statistics.mean(vals) if vals else None


def summary(rows):
    s = {"n": len(rows), "stop": sum(r["ended"] == "stop token" for r in rows) / len(rows),
         "repetition": sum(repetition(r["output"]) for r in rows) / len(rows),
         "devanagari": sum(bool(re.search(r"[ऀ-ॿ]", r["output"])) for r in rows) / len(rows),
         "chrf": statistics.mean(best_chrf(r) for r in rows), "persona_overlap": overlap(rows),
         "median_tokens": statistics.median(r["new_tokens"] for r in rows)}
    return {k: round(v, 3) if isinstance(v, float) else v for k, v in s.items()}


SUMMARY = {f"{split} / {mem}": summary([r for r in RESULTS if r["split"] == split and r["memory"] == mem])
           for split, mem in [("validation", "right"), ("validation", "other"), ("test", "right"), ("ood", "right"),
                              ("minimal_pairs", "right")]}
ref_overlap = overlap([{"source_id": r["source_id"], "output": r["target"]} for r in VAL])
for k, v in SUMMARY.items():
    print(f"{k:26s} {v}")
print(f"persona overlap of the validation references: {ref_overlap:.3f}")

right = {r["id"]: r for r in RESULTS if r["split"] == "validation" and r["memory"] == "right"}
other = {r["id"]: r for r in RESULTS if r["split"] == "validation" and r["memory"] == "other"}
changed = sum(right[i]["output"] != other[i]["output"] for i in right) / len(right)
chrf_drop = SUMMARY["validation / right"]["chrf"] - SUMMARY["validation / other"]["chrf"]
print(f"\nmemory dependence: outputs changed by another sentence's memory {changed:.0%} | "
      f"chrF right {SUMMARY['validation / right']['chrf']:.1f} vs other {SUMMARY['validation / other']['chrf']:.1f} "
      f"(drop {chrf_drop:.1f})")

mp = {}
for r in RESULTS:
    if r["split"] == "minimal_pairs":
        mp.setdefault((r["pair_id"], r["persona"]), {})[r["source_id"]] = r
differ, number_ok, number_n = 0, 0, 0
for (pid, persona), d in mp.items():
    a, b = [d[k] for k in sorted(d)]
    differ += a["output"] != b["output"]
    na, nb = re.findall(r"\d+", a["source"]), re.findall(r"\d+", b["source"])
    if na != nb:
        number_n += 1
        number_ok += all(n in re.findall(r"\d+", a["output"]) for n in na) and \
            all(n in re.findall(r"\d+", b["output"]) for n in nb)
MP = {"outputs differ": round(differ / len(mp), 3), "number pairs carried": f"{number_ok}/{number_n}"}
print(f"minimal pairs: outputs differ within a pair {MP['outputs differ']:.0%} | numbers carried {MP['number pairs carried']}")

val = SUMMARY["validation / right"]
last_norms = LOG[-1].get("grad_norms", {})
GATE = {
    "1 all five groups train": bool(last_norms) and all(v > 0 for v in last_norms.values()),
    "2 no degenerate output": LOG[1]["train_loss"] > LOG[-1]["train_loss"] and val["stop"] >= 0.95
                              and val["repetition"] <= 0.05,
    "3 decoder uses the source through the bridge": LOG[BEST_EPOCH]["validation"]["gap"] >= 0.1 and chrf_drop >= 2,
    "4 minimal pairs": differ / len(mp) >= 0.8 and (number_n == 0 or number_ok / number_n >= 0.8),
}
print("\nPHASE 3 GATE")
for k, ok in GATE.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {k}")

validation / right         {'n': 75, 'stop': 1.0, 'repetition': 0.027, 'devanagari': 0.0, 'chrf': 15.672, 'persona_overlap': 0.217, 'median_tokens': 13}
validation / other         {'n': 75, 'stop': 1.0, 'repetition': 0.027, 'devanagari': 0.0, 'chrf': 15.485, 'persona_overlap': 0.217, 'median_tokens': 13}
test / right               {'n': 75, 'stop': 1.0, 'repetition': 0.053, 'devanagari': 0.0, 'chrf': 14.497, 'persona_overlap': 0.196, 'median_tokens': 13}
ood / right                {'n': 114, 'stop': 0.991, 'repetition': 0.009, 'devanagari': 0.0, 'chrf': 13.876, 'persona_overlap': 0.207, 'median_tokens': 13.0}
minimal_pairs / right      {'n': 72, 'stop': 1.0, 'repetition': 0.0, 'devanagari': 0.0, 'chrf': 14.25, 'persona_overlap': 0.251, 'median_tokens': 17.0}
persona overlap of the validation references: 0.338

memory dependence: outputs changed by another sentence's memory 89% | chrF right 15.7 vs other 15.5 (drop 0.2)
minimal pairs: outputs differ within a pair 56% | numbers carried 0

## Cell 11 — Validation outputs side by side

For each validation source: the English, then per persona the output with the right memory, with another sentence's memory, and the first reference.

In [12]:
# Cell 11 — Validation: right memory, other memory, reference

for sid in sorted({r["source_id"] for r in VAL}):
    rows = [r for r in VAL if r["source_id"] == sid]
    print(f"\n=== {sid} English: {rows[0]['source']}")
    for r in rows:
        a, b = right[r["id"]], other[r["id"]]
        flag = "" if a["ended"] == "stop token" else "  [NO STOP]"
        print(f"  {r['persona']}")
        print(f"    right     | {a['output']}{flag}")
        print(f"    other     | {b['output']}   (memory of: {b['memory_of'][:60]})")
        print(f"    reference | {r['references'][0]}")


=== s041 English: We should celebrate the progress without ignoring the remaining work.
  don
    right     | Blue folder mein original file hai.
    other     | Farah ka laptop use karne se pehle permission chahiye.   (memory of: I will support the decision once everyone understands the tr)
    reference | Progress ki khushi manao. Par baaki kaam ko nazar se mat hatao.
  anthony_gonsalves
    right     | Blue folder ko green folder ka replacement mat karo, warna file ka track lost ho jaayega!
    other     | Blue folder ko Monday ko return kar do, warna green folder ko bhi koach kar dena!   (memory of: I will support the decision once everyone understands the tr)
    reference | Progress ki party banti hai, lekin baaki kaam mat bhoolna, cake ke baad bartan bhi hain!
  jai_sholay
    right     | Farah ka laptop use karne se pehle permission chahiye.
    other     | Farah ka laptop use karne se pehle permission chahiye.   (memory of: I will support the decision once everyone understand

## Cell 12 — Save the trained parts

Encoder LoRA and decoder LoRA (`save_pretrained`), and the bridge with the cross-attention blocks and gates (`torch.save`), under `/kaggle/working/phase3/`. Files in `/kaggle/working` are lost when an interactive session ends: download the folder (or save a version) if this checkpoint is to be used in Phase 4.

In [13]:
# Cell 12 — Save both LoRAs and the bridge + cross-attention

import os

OUT_DIR = "/kaggle/working/phase3"
os.makedirs(OUT_DIR, exist_ok=True)
encoder.save_pretrained(f"{OUT_DIR}/encoder-lora")
dec_model.save_pretrained(f"{OUT_DIR}/decoder-lora")
torch.save({"bridge": bridge.state_dict(), "xattn": xattn.state_dict(), "xattn_layers": XATTN_LAYERS,
            "heads": XATTN_HEADS, "d_enc": D_ENC, "d_dec": D_DEC}, f"{OUT_DIR}/bridge_xattn.pt")
!du -sh /kaggle/working/phase3/*

808M	/kaggle/working/phase3/bridge_xattn.pt
114M	/kaggle/working/phase3/decoder-lora
13M	/kaggle/working/phase3/encoder-lora


## Cell 13 — Settings and every record

In [14]:
# Cell 13 — Settings, logs and all outputs as JSON lines

SETTINGS = {
    "versions": VERSIONS, "data": {"url": DATA, "counts": {s: len(rs) for s, rs in SPLITS.items()}},
    "encoder": {"model": ENCODER, "lora": {"r": 16, "alpha": 32, "targets": ["Wqkv", "Wo", "Wi"]}},
    "decoder": {"model": DECODER, "precision": "4-bit NF4, fp32 compute", "lora": {"r": 16, "alpha": 32, "targets": PROJ}},
    "bridge": "Linear (no bias) + RMSNorm",
    "cross_attention": {"after_layers": XATTN_LAYERS, "heads": XATTN_HEADS, "bias": False, "gate": "tanh, from 0"},
    "stage_a": {"steps": WARMUP_STEPS, "log": WARMUP_LOG},
    "stage_b": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "micro_batch": MICRO, "accumulation": ACCUM,
                "lrs": LRS, "schedule": "linear decay", "kept_epoch": BEST_EPOCH, "seconds": TRAIN_SECONDS,
                "peak_memory_gb": PEAK, "log": LOG},
    "summary": SUMMARY, "reference_overlap": ref_overlap, "memory_dependence": {"changed": changed, "chrf_drop": chrf_drop},
    "minimal_pairs": MP, "gate": GATE, "checks": CHECKS,
}
print("----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1, default=str))
print("\n----- RECORDS (JSON lines) -----")
for r in RESULTS:
    print(json.dumps(r, ensure_ascii=False))
print("----- END RECORDS -----")

----- SETTINGS -----
{
 "versions": {
  "torch": "2.11.0+cu128",
  "transformers": "5.17.0",
  "peft": "0.21.2",
  "bitsandbytes": "0.50.2",
  "gpus": [
   "Tesla T4",
   "Tesla T4"
  ],
  "revisions": {
   "answerdotai/ModernBERT-base": "8949b909ec900327062f0ebf497f51aef5e6f0c8",
   "google/gemma-3-4b-it": "093f9f388b31de276ce2de164bdc2081324b9767"
  }
 },
 "data": {
  "url": "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/persona_translation_v4/merged/",
  "counts": {
   "train": 768,
   "validation": 75,
   "test": 75,
   "ood": 114,
   "minimal_pairs": 72
  }
 },
 "encoder": {
  "model": "answerdotai/ModernBERT-base",
  "lora": {
   "r": 16,
   "alpha": 32,
   "targets": [
    "Wqkv",
    "Wo",
    "Wi"
   ]
  }
 },
 "decoder": {
  "model": "google/gemma-3-4b-it",
  "precision": "4-bit NF4, fp32 compute",
  "lora": {
   "r": 16,
   "alpha": 32,
   "targets": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    

---
## Next

Save the notebook (clear the token in Cell 0 first) and download `/kaggle/working/phase3/` if the gate passes.

- **Gate passes:** Phase 4 — the encoder interventions (full, null encoder, mismatched source) against the direct-source decoder baseline trained on the same v4 data, with the parameter-matched control.
- **Gate 3 fails** (the decoder does not use the source): the bridge has not learned alignment from 768 pairs; plan proposal P1 — warm the bridge up on a public English → Hinglish parallel corpus — is the next experiment.
- **Gates 1, 2 or 4 fail:** fix those first (training setup, stopping, minimal pairs) before judging the bridge.